# KJH data diagnostics
Adapted from origin/kjh_dashboard for the final 9-input submission data.


## Reproduce preprocessing


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/okm_cleaned_2021.csv").is_file())
sys.path.insert(0, str(ROOT))
from scripts import build_eda_evidence as eda
RAW_PATH = ROOT / "data/okm_augumented_2021.csv"
CLEAN_PATH = ROOT / "data/okm_cleaned_2021.csv"
FIGURES = ROOT / "report/figures"
EVIDENCE = ROOT / "report/evidence"
raw, clean = eda.load_data()
from src.preprocessing import run_preprocessing_pipeline
cleaned_file = pd.read_csv(CLEAN_PATH, encoding="utf-8-sig")


## Full time series


In [ ]:
summary = eda.build()
display(Image(filename=str(FIGURES / "08_eda_overview.png")))


## Cleaned CSV comparison


In [ ]:
recreated = run_preprocessing_pipeline(data_path=RAW_PATH, save_summary=False)
provided = pd.read_csv(CLEAN_PATH, encoding="utf-8-sig")
recreated[eda.DATE] = pd.to_datetime(recreated[eda.DATE]).dt.strftime("%Y-%m-%d")
pd.testing.assert_frame_equal(recreated.reset_index(drop=True), provided.reset_index(drop=True),
                              check_dtype=False, rtol=1e-10, atol=1e-10)
print("Recreated cleaned data matches the provided CSV.")


## Missing values and shutdown


In [ ]:
display(Image(filename=str(FIGURES / "09_eda_quality.png")))
display(pd.DataFrame({"raw_missing": raw.isna().sum(), "cleaned_missing": clean.isna().sum()}).fillna(0))


## Weekday and hour


In [ ]:
display(Image(filename=str(FIGURES / "10_eda_heatmap.png")))
print("Weekday 11:00 and 12:00 mean kW:", summary["weekday_11_kw"], summary["weekday_12_kw"])


## Restored timestamps


In [ ]:
print("Restored hour labels:", summary["restored_hours"])
display(clean.loc[clean[eda.RESTORED], [eda.DATE, eda.HOUR, "datetime"]].head(10))


## Operating modes


In [ ]:
display(Image(filename=str(FIGURES / "11_eda_operating_modes.png")))
print("Zero-production non-shutdown hours:", summary["idle_hours"])
print("Mean idle power (kW):", summary["idle_mean_kw"])


## Summary


In [ ]:
display(clean[[eda.POWER, eda.PRODUCTION]].describe())
print("Shutdown hours:", summary["shutdown_hours"], "Remaining missing values:", summary["clean_missing"])
